In [1]:
import scanpy as sc
import squidpy as sq
import pandas as pd
import matplotlib.pyplot as plt
import os

/home/ec2-user/miniforge3/envs/spatial/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [15]:
table_dir = os.path.expanduser("../results/tables")
figure_dir = os.path.expanduser("../results/figures") 

In [3]:
adata = sc.read_h5ad(os.path.expanduser(
    "../data/processed/02_preprocessed.h5ad"))

In [5]:
# Moran's I
genes = adata.var_names[adata.var["highly_variable"]].tolist()

sq.gr.spatial_autocorr(
    adata,
    mode="moran",
    genes=genes,
    n_perms=100, 
    n_jobs=4     
)

100%|███████████████████████████████████████████████████████████████████████████████████| 100/100 [00:32<00:00,  3.10/s]


In [6]:
moran_df = adata.uns["moranI"].copy()
moran_df = moran_df.sort_values("I", ascending=False)
print("Top 20 SVGs by Moran's I:")
print(moran_df.head(20)[["I", "pval_norm_fdr_bh"]])

Top 20 SVGs by Moran's I:
                 I  pval_norm_fdr_bh
MBP       0.807505               0.0
PLP1      0.701897               0.0
SCGB2A2   0.701657               0.0
MOBP      0.646056               0.0
GFAP      0.624622               0.0
TF        0.592670               0.0
CNP       0.571668               0.0
CRYAB     0.561345               0.0
MAG       0.545858               0.0
PPP1R14A  0.533418               0.0
ERMN      0.508906               0.0
CLDND1    0.500297               0.0
CLDN11    0.495613               0.0
SCGB1D2   0.492273               0.0
MOG       0.477463               0.0
SPP1      0.471823               0.0
S100B     0.459951               0.0
ENC1      0.458089               0.0
NRGN      0.456248               0.0
SNAP25    0.444846               0.0


In [7]:
moran_df.to_csv(os.path.join(table_dir, "moran_I_results.csv"))

In [14]:
known_markers = {
    "PCP4": "L5",
    "MOBP": "WM",
    "MBP": "WM",
    "SNAP25": "Gray matter",
    "RELN": "L1"
}

print("\nRank of known markers:")
for gene, layer in known_markers.items():
    if gene in moran_df.index:
        rank = moran_df.index.get_loc(gene) + 1
        I_val = moran_df.loc[gene, "I"]
        print(f"  {gene} ({layer}): rank {rank}, I={I_val:.4f}")
    else:
        print(f"  {gene}: not in HVG list")
        if gene in adata.var_names:
            sq.gr.spatial_autocorr(adata, mode="moran", genes=[gene], n_perms=100)


Rank of known markers:
  PCP4 (L5): rank 67, I=0.2986
  MOBP (WM): rank 4, I=0.6461
  MBP (WM): rank 1, I=0.8075
  SNAP25 (Gray matter): rank 20, I=0.4448
  RELN (L1): rank 399, I=0.1028


In [18]:
# top SVGs
top_svgs = moran_df.head(10).index.tolist()

top_svgs = [g for g in top_svgs if g in adata.var_names]

fig, axes = plt.subplots(2, 5, figsize=(25, 10))
for i, gene in enumerate(top_svgs):
    ax = axes[i // 5, i % 5]
    sc.pl.spatial(adata, color=gene, ax=ax, show=False,
                  title=f"{gene}\nI={moran_df.loc[gene, 'I']:.3f}",
                  spot_size=100)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, "top10_svgs_spatial.png"), dpi=200, bbox_inches='tight')
plt.close()


markers_in_data = [g for g in known_markers.keys() if g in adata.var_names]
fig, axes = plt.subplots(1, len(markers_in_data), figsize=(5*len(markers_in_data), 5))
for i, gene in enumerate(markers_in_data):
    ax = axes[i] if len(markers_in_data) > 1 else axes
    sc.pl.spatial(adata, color=gene, ax=ax, show=False,
                  title=f"{gene} ({known_markers[gene]})", spot_size=100)
plt.tight_layout()
plt.savefig(os.path.join(figure_dir, "known_markers_spatial.png"), dpi=200, bbox_inches='tight')
plt.close()

/tmp/ipykernel_3837/3673811242.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color=gene, ax=ax, show=False,
/tmp/ipykernel_3837/3673811242.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color=gene, ax=ax, show=False,
/tmp/ipykernel_3837/3673811242.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color=gene, ax=ax, show=False,
/tmp/ipykernel_3837/3673811242.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color=gene, ax=ax, show=False,
/tmp/ipykernel_3837/3673811242.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color=gene, ax=ax, show=False,
/tmp/ipykernel_3837/3673811242.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(adata, color=gene, ax=ax, show=False,
/tmp/ipykernel_3837/3673811242.py:9: FutureWarning: Use `squidpy.pl.spatial_scatter` instead.
  sc.pl.spatial(ad